# L02 · CE, entropy, KL, and KD

## Goal

**Estimated time:** 40 min · **Path:** fast, full

- compute CE, entropy, and KL
- name the KL direction
- verify generalized-JSD boundaries

### Current position: L01 → **L02** → L03

```text
Prompt/Data -> state source -> ... -> L02 -> ... -> fair evaluation
```

Alt text: The course map highlights L02 between its prerequisite and next lesson; every method remains connected to the same evaluation stage.

## Setup

In [1]:
LESSON_ID = "L02"
from pathlib import Path
import sys
import torch

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root / "src"))

import opd_study
from opd_study.device import resolve_device
from opd_study.utils import seed_everything

seed_everything(42)
device_report = resolve_device("cpu")
print({"lesson": LESSON_ID, "opd_study": opd_study.__version__,
       "torch": torch.__version__, "device": device_report.selected,
       "profile": "toy", "network": "not required"})

{'lesson': 'L02', 'opd_study': '0.1.0.dev0', 'torch': '2.13.0', 'device': 'cpu', 'profile': 'toy', 'network': 'not required'}


## Steps

### 1/3 · 8–12 min

KL is not a symmetric distance. This course fixes forward as KL(teacher||student) and reverse as KL(student||teacher), and names both distributions in APIs.

Figure alt: labels and numbers remain readable without color.

### Core mechanics

Hard-label CE retains one target token; KD retains the teacher's full distribution. `KL(teacher || student)` heavily penalizes missing teacher-supported regions, while `KL(student || teacher)` heavily penalizes student mass in teacher-low-probability regions. Coverage- versus mode-seeking is a useful tendency, not a guaranteed behavior law.

Temperature `T` divides logits to soften distributions. Classical KD multiplies by `T²` to compensate the gradient-scale change. This repository computes log-softmax in float32 and rejects empty masks or nonpositive temperatures.

### Production implementation: why this design

Every KL API names its arguments `(teacher_logits, student_logits)`. Before reduction the shape is `[B,T]`; only response positions enter the final mean. Generalized-JSD beta boundaries use explicit branches to avoid `log(0)`.

Production code: [`math.py`](../../src/opd_study/math.py), [`losses.py`](../../src/opd_study/algorithms/losses.py).

In [2]:
import inspect
from opd_study.math import forward_kl_from_logits, reverse_kl_from_logits

objects_to_show = (forward_kl_from_logits, reverse_kl_from_logits,)
for object_to_show in objects_to_show:
    source_lines = inspect.getsource(object_to_show).splitlines()
    print(f"\n# {object_to_show.__module__}.{object_to_show.__qualname__}")
    print("\n".join(source_lines[:80]))
    if len(source_lines) > 80:
        print(f"... {len(source_lines) - 80} more lines; open the linked source file")


# opd_study.math.forward_kl_from_logits
def forward_kl_from_logits(
    teacher_logits: Tensor,
    student_logits: Tensor,
    *,
    temperature: float = 1.0,
) -> Tensor:
    """Return ``KL(teacher || student)`` without reducing token/batch dimensions."""

    _validate_logits(teacher_logits, student_logits)
    teacher_log_p = log_probs(teacher_logits, temperature=temperature)
    student_log_p = log_probs(student_logits, temperature=temperature)
    return (teacher_log_p.exp() * (teacher_log_p - student_log_p)).sum(dim=-1)

# opd_study.math.reverse_kl_from_logits
def reverse_kl_from_logits(
    teacher_logits: Tensor,
    student_logits: Tensor,
    *,
    temperature: float = 1.0,
) -> Tensor:
    """Return ``KL(student || teacher)`` without reducing token/batch dimensions."""

    _validate_logits(teacher_logits, student_logits)
    teacher_log_p = log_probs(teacher_logits, temperature=temperature)
    student_log_p = log_probs(student_logits, temperature=temperature)
    retur

### Alternatives and trade-offs

When full logits are expensive, store top-k logits or sampled tokens. Top-k discards tail mass, so report retained probability mass and approximation error. If an API teacher exposes only partial log-probabilities, do not label the result full KL.

### 2/3 · Run and observe

Predict before running: which invariant should you inspect first in L02's output? Write one sentence, then run.

In [3]:
from opd_study.math import (entropy_from_logits, forward_kl_from_logits,
                            generalized_jsd_from_logits, reverse_kl_from_logits)

teacher = torch.log(torch.tensor([[0.70, 0.20, 0.10]]))
student = torch.log(torch.tensor([[0.40, 0.35, 0.25]]))
values = {
    "H(teacher)": entropy_from_logits(teacher).item(),
    "KL(teacher||student)": forward_kl_from_logits(teacher, student).item(),
    "KL(student||teacher)": reverse_kl_from_logits(teacher, student).item(),
    "JSD_beta=.5": generalized_jsd_from_logits(teacher, student, beta=.5).item(),
}
print({name: round(value, 5) for name, value in values.items()})

{'H(teacher)': 0.80182, 'KL(teacher||student)': 0.18818, 'KL(student||teacher)': 0.20109, 'JSD_beta=.5': 0.04768}


In [4]:
boundaries = [generalized_jsd_from_logits(teacher, student, beta=beta).item()
              for beta in (0.0, 0.25, 0.5, 0.75, 1.0)]
print("beta sweep:", [round(value, 5) for value in boundaries])
print("Argument order is part of the definition; KL is not symmetric.")

beta sweep: [0.18818, 0.03537, 0.04768, 0.0365, 0.20109]
Argument order is part of the definition; KL is not symmetric.


## Checks

In [5]:
assert values["KL(teacher||student)"] >= 0
assert values["KL(student||teacher)"] >= 0
assert abs(boundaries[0] - values["KL(teacher||student)"]) < 1e-6
assert abs(boundaries[-1] - values["KL(student||teacher)"]) < 1e-6
print("check passed: non-negativity and named beta boundaries")

check passed: non-negativity and named beta boundaries


**Exercise (7 min):** hand-compute FKL and RKL for teacher `[0.99,0.01]`, student `[0.5,0.5]`; explain why swapping arguments changes the value.

<details><summary>Check</summary>Compute the teacher-weighted and student-weighted log-ratios separately and name KL asymmetry.</details>

## My recurring mistakes

### M1 — Memorizing unnamed `KL(p,q)` direction

- Wrong: guess roles from argument position.
- Why: paper/library conventions vary.
- Fix: write `KL(teacher || student)` with named arguments.
- Related check: `test_forward_kl_matches_hand_calculation`

### M2 — Comparing temperatures without gradient scaling

- Wrong: interpret loss-size changes without noting `T²` correction.
- Why: softmax derivative scale also changes.
- Fix: record temperature and correction convention in the run card.
- Related check: `test_temperature_and_empty_masks_fail_loudly`

## 60-second summary

1. compute CE, entropy, and KL
2. name the KL direction
3. verify generalized-JSD boundaries

## Next Steps

Before the next notebook, rerun the assertions and record one prediction you revised.

### Sources

- [`gkd`](https://arxiv.org/abs/2306.13649v3) · `2306.13649v3` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)